In [3]:
import pandas as pd
import numpy as np
import pymongo
import os


data = pd.read_csv('healthcare_dataset.csv')



In [4]:
def verify_data(data):
    print("Aperçu des données :")
    print(data.head())
    print("\nInformations sur les données :")
    print(data.info())
    print("\nStatistiques descriptives :")
    print(data.describe())

In [5]:
verify_data(data)

Aperçu des données :
            Name  Age  Gender Blood Type Medical Condition Date of Admission  \
0  Bobby JacksOn   30    Male         B-            Cancer        2024-01-31   
1   LesLie TErRy   62    Male         A+           Obesity        2019-08-20   
2    DaNnY sMitH   76  Female         A-           Obesity        2022-09-22   
3   andrEw waTtS   28  Female         O+          Diabetes        2020-11-18   
4  adrIENNE bEll   43  Female        AB+            Cancer        2022-09-19   

             Doctor                    Hospital Insurance Provider  \
0     Matthew Smith             Sons and Miller         Blue Cross   
1   Samantha Davies                     Kim Inc           Medicare   
2  Tiffany Mitchell                    Cook PLC              Aetna   
3       Kevin Wells  Hernandez Rogers and Vang,           Medicare   
4    Kathleen Hanna                 White-White              Aetna   

   Billing Amount  Room Number Admission Type Discharge Date   Medication  \


In [15]:
def clean_data(data):
    # Supprimer les doublons
    data = data.drop_duplicates()
    data = data.dropna()
    data['Name'] = data['Name'].str.title()
    data['Name'] = data['Name'].str.strip()
    data['Billing Amount'] = data['Billing Amount'].round(2)
    return data

In [ ]:
cleaned_data = clean_data(data)

In [19]:
data

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,eLIZABeTH jaCkSOn,42,Female,O+,Asthma,2020-08-16,Joshua Jarvis,Jones-Thompson,Blue Cross,2650.714952,417,Elective,2020-09-15,Penicillin,Abnormal
55496,KYle pEREz,61,Female,AB-,Obesity,2020-01-23,Taylor Sullivan,Tucker-Moyer,Cigna,31457.797307,316,Elective,2020-02-01,Aspirin,Normal
55497,HEATher WaNG,38,Female,B+,Hypertension,2020-07-13,Joe Jacobs DVM,"and Mahoney Johnson Vasquez,",UnitedHealthcare,27620.764717,347,Urgent,2020-08-10,Ibuprofen,Abnormal
55498,JENniFER JOneS,43,Male,O-,Arthritis,2019-05-25,Kimberly Curry,"Jackson Todd and Castro,",Medicare,32451.092358,321,Elective,2019-05-31,Ibuprofen,Abnormal


In [ ]:

def import_to_mongodb(data):
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client['DataSoluTech']
    chunk_size = 1000
    num_chunks = int((len(cleaned_data))/chunk_size)
    chunks = []
        
    for i in range(num_chunks):
        start = chunk_size * i
        stop = start + chunk_size
        chunks.append(cleaned_data[start:stop])
    # itérer sur les blocs    
    for i in range(num_chunks):
        db.healthcare.insert_many(chunks[i].to_dict('records'))
        print(f"Chunk {i+1}/{num_chunks} imported successfully.")
import_to_mongodb(cleaned_data)

Chunk 1/55 imported successfully.
Chunk 2/55 imported successfully.
Chunk 3/55 imported successfully.
Chunk 4/55 imported successfully.
Chunk 5/55 imported successfully.
Chunk 6/55 imported successfully.
Chunk 7/55 imported successfully.
Chunk 8/55 imported successfully.
Chunk 9/55 imported successfully.
Chunk 10/55 imported successfully.
Chunk 11/55 imported successfully.
Chunk 12/55 imported successfully.
Chunk 13/55 imported successfully.
Chunk 14/55 imported successfully.
Chunk 15/55 imported successfully.
Chunk 16/55 imported successfully.
Chunk 17/55 imported successfully.
Chunk 18/55 imported successfully.
Chunk 19/55 imported successfully.
Chunk 20/55 imported successfully.
Chunk 21/55 imported successfully.
Chunk 22/55 imported successfully.
Chunk 23/55 imported successfully.
Chunk 24/55 imported successfully.
Chunk 25/55 imported successfully.
Chunk 26/55 imported successfully.
Chunk 27/55 imported successfully.
Chunk 28/55 imported successfully.
Chunk 29/55 imported successf

In [ ]:
def mongoConnection():
    try:
     # Fetching client using pymongo
        client = pymongo.MongoClient(mongoUrl)
        # Holding our database
        mydatabase = client[dbName]
        return mydatabase
    except Exception as e:
        print("Error occured while connecting to database!")
        return False


In [ ]:
def mongoImport(data, dbName, collectionName):
    try:
        mydatabase = mongoConnection()
        if mydatabase:
            collection = mydatabase[collectionName]
            collection.bulk_write(data.to_dict('records'))
            print("Data imported successfully!")
        else:
            print("Failed to connect to database. Data import aborted.")
    except Exception as e:
        print("Error occured while importing data to MongoDB!")

In [ ]:
db_schema = {
    'Name': {
        'type': 'string',
        'minlength': 1,
        'required': True,
    },
    'Age': {
        'type': 'int',
        'minlength': 1,
        'required': True,
    },
    'Gender': {
        'type': 'string',
        "required": False,
        'enum': ['Male', 'Female', 'Other']
    },
    'Blood Type': {
        'type': 'string',
        'required': True,
        'enum': ['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-']
    },
    'Medical Condition': {
        'type': 'string',
        'required': True,
    },
    'Date of Admission': {
        'type': 'date',
        'required': True,
    },
    'Doctor': {
        'type': 'string',
        'required': True,
    },
    "Hospital": {
        "type": "string",
        "required": True
    },
    'Insurance Provider': {
        'type': 'string',
        'required': True,
    },
    'Billing Amount': {
        'type': 'float',
        'required': True,
    },
    'Billing Amount': {
        'type': 'float',
        'required': True,
    },
    'Billing Amount': {
        'type': 'float',
        'required': True,
    },
    'Room Number': {
        'type': 'int',
        'required': True,
    },
    'Admission Type': {
        'type': 'string',
        'required': True,
        'enum': ['Emergency', 'Elective', 'Urgent']
    },
    'Discharge Date': {
        'type': 'date',
        'required': True,
    },
    'Medication': {
        'type': 'string',
        'required': True,
    },
     'Test Results': {
        'type': 'string',
        'required': True,
        'enum' : ['Normal', 'Abnormal', 'Inconclusive']
    },
}
                
